# F, G, H. 평가 종합 · 보고서 생성 · 보고서 검증

| | F. 평가 종합 | G. 보고서 생성 | H. 보고서 검증 |
|---|---|---|---|
| **담당** | LLM (검색 없음) | LLM (검색 없음) | 규칙 기반 (LLM 없음) |
| **선행 노드** | C, D, E | F (또는 H 재진입) | G |
| **출력** | `synthesis` | `final_report` | `validation_result`, `retry_count` |

셋이 순서대로 이어지고(F→G→H) H가 실패하면 G로 되돌아가는 유일한 루프라 한 노트북에 같이 둔다.
H는 LLM을 안 쓰는 규칙 기반 노드라는 게 포인트 — 챕터 존재 여부·서열 표현 포함 여부를 문자열 검사로만 판단한다.

이 노트북 끝에서 만든 것들은 `src/nodes_fgh.py`로 저장된다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import config, prompts
from src.schemas import Label, ValidationResult

## 1. F. 평가 종합 — 프롬프트 확인

In [ ]:
print(prompts.SYNTHESIS_PROMPT)

## 2. F. 노드 함수 정의

`market_eval`/`stakeholder_eval`/`domain_eval`은 C/D/E가 각자 쓴 State 키에서 읽고,
TRL은 `tech_research`(B가 씀) 안에 있어서 여기서 따로 뽑아낸다 — TRL 전담 에이전트가 없기 때문(2-1절).

In [ ]:
def make_node_f(llm):
    """F. 평가 종합. 4관점(시장/이해관계자/도메인/TRL) 라벨을 모아 비교한다.

    labels 스키마를 여기서 고정해 만든다 — 공유 파일 schemas.py 의
    Synthesis.labels 가 dict[str, str](자유 형식 object)라 두 가지가 깨진다.
      1) OpenAI 기본 strict 구조화 출력(json_schema)이 400 으로 거부한다.
         "'required' ... must include every key in properties"
      2) method="function_calling" 으로 우회하면 호출마다 모양이 달라진다.
         실측: 1회차는 한국어 키 dict, 2회차는 list 가 돌아와 ValidationError.
    schemas.py 를 건드리지 않으려고 create_model 로 고정 스키마를 만들어
    LLM 에 넘기고, State 에는 기존과 똑같은 dict 모양으로 되돌려 넣는다.
    (근본 해결은 schemas.py 의 labels 를 고정 필드로 바꾸는 것 — 조 합의 필요)
    클래스 문으로 안 쓰는 이유: Jupyter 셀에서 정의한 클래스는
    inspect.getsource 가 못 읽어 '파일로 저장' 셀이 깨진다.
    """
    from pydantic import create_model

    labels_model = create_model(
        "SynthesisLabels",
        market=(Label, ...),
        stakeholder=(Label, ...),
        domain=(Label, ...),
        trl=(Label, ...),
    )
    strict_model = create_model(
        "SynthesisStrict",
        labels=(labels_model, ...),
        conflicts=(list[str], ...),
        reasoning=(str, ...),
    )
    structured_llm = llm.with_structured_output(strict_model)

    def node_f_synthesis(state):
        tech_research = state.get("tech_research", {})
        trl_eval = {
            name: r.get("trl_assessment", {}) for name, r in tech_research.items()
        }
        prompt = prompts.SYNTHESIS_PROMPT.format(
            market_eval=state.get("market_eval", {}),
            stakeholder_eval=state.get("stakeholder_eval", {}),
            domain_eval=state.get("domain_eval", {}),
            trl_eval=trl_eval,
        )
        result = structured_llm.invoke(prompt)
        # State 모양은 기존 Synthesis.model_dump() 와 동일하게 유지한다.
        return {
            "synthesis": {
                "labels": result.labels.model_dump(),
                "conflicts": result.conflicts,
                "reasoning": result.reasoning,
            }
        }

    return node_f_synthesis

## 3. G. 보고서 생성 — 프롬프트 확인

4개로 나뉜 `*_references`를 여기서 하나로 합친다(reducer가 아니라 그냥 리스트 덧셈).
분석 배경은 State가 아니라 `config.ANALYSIS_BACKGROUND` 고정 문단을 그대로 쓴다.

In [ ]:
print(prompts.REPORT_PROMPT)

In [ ]:
def make_node_g(llm):
    """G. 보고서 생성. H가 무효 판정을 내리면 재진입해서 revision_note를 받는다."""
    def node_g_report(state):
        validation = state.get("validation_result")
        revision_note = ""
        if validation and not validation.get("is_valid", True):
            items = validation.get("missing_items", [])
            violations = [i for i in items if i.startswith("서열 표현")]
            formats = [i for i in items if i.startswith(FORMAT_PREFIX)]
            absent = [i for i in items if i not in violations and i not in formats]
            parts = []
            if absent:
                parts.append(
                    f"다음 장이 빠졌다: {', '.join(absent)}. 이번엔 반드시 포함하라."
                )
            if formats:
                parts.append(
                    "형식 규칙을 어겼다 — "
                    + " / ".join(f[len(FORMAT_PREFIX):] for f in formats)
                )
            if violations:
                parts.append(
                    f"다음이 본문에 들어 있다: {', '.join(violations)}. "
                    "해당 표현을 지우거나 중립 서술로 바꿔라. "
                    "단 금칙어를 설명하는 문장 자체도 쓰지 말 것."
                )
            revision_note = "[재작성 지시] " + " ".join(parts)

        all_references = (
            state.get("tech_references", [])
            + state.get("market_references", [])
            + state.get("stakeholder_references", [])
            + state.get("domain_references", [])
        )

        # F가 만든 라벨만 넘기면 E가 뽑은 GB·%p·ms·W 수치와 각 관점의 notes가
        # 보고서에 도달하지 않는다. REPORT_PROMPT(공유 파일)를 고치지 않고
        # {synthesis} 슬롯에 종합과 원자료를 함께 실어 보낸다.
        perspective_block = {
            "관점 간 종합(F)": state.get("synthesis", {}),
            "시장성 원자료(C)": state.get("market_eval", {}),
            "이해관계자 원자료(D)": state.get("stakeholder_eval", {}),
            "도메인 원자료(E)": state.get("domain_eval", {}),
        }

        prompt = prompts.REPORT_PROMPT.format(
            analysis_background=config.ANALYSIS_BACKGROUND,
            selected_technologies=state.get("selected_technologies", {}),
            tech_research=state.get("tech_research", {}),
            synthesis=perspective_block,
            references=all_references,
            revision_note=revision_note,
        )
        report_text = llm.invoke(prompt).content
        return {"final_report": report_text}

    return node_g_report

## 4. H. 보고서 검증 — 규칙 정의

LLM을 안 부른다. `REQUIRED_CHAPTERS`가 전부 본문에 있는지, `FORBIDDEN_WORDS`(서열 표현)가 안 섞였는지만 문자열로 검사한다.
`retry_count`가 `config.MAX_RETRY_H`(2)를 넘으면 강제로 통과시키되 `forced_pass=True`로 표시한다 — G가 이걸 보고 「한계점」 장에 뭐가 빠졌는지 적는다.

In [ ]:
REQUIRED_CHAPTERS = ['SUMMARY', '시장', '이해관계자', '도메인', 'REFERENCE']
FORBIDDEN_WORDS = ['우수', '우월', '우위', '권장', '추천']
RANKING_CHECK_CHAPTERS = ['관점별 평가', '시장', '이해관계자', '도메인', '시사점']

# 아래 상수의 뜻과 근거는 _check_format 독스트링에 있다.
# 저장 셀이 상수를 값만(!r) 써서 여기 주석은 .py 로 넘어가지 않기 때문이다.
PERSPECTIVE_CHAPTERS = ['시장', '이해관계자', '도메인']
MIN_SUMMARY_BLOCKS = 3
MIN_BULLET_LINES = 15
MIN_NUMBERED_REFS = 5
FORMAT_PREFIX = '형식 위반: '


def _check_format(report, chapters):
    """개조식·[소결]·번호 인용이 실제로 지켜졌는지 본다.

    장 제목 유무만 보던 이전 판은 "### 시장 관점" 같은 줄글 보고서를 첫 판에
    통과시켰다. REPORT_PROMPT가 요구하는 개조식·[소결]·번호 인용이 실제로
    지켜졌는지까지 봐야 G<->H 루프가 형식 드리프트를 교정할 수 있다.

    임계값 - MIN_SUMMARY_BLOCKS: 4.1/4.2/4.3 각 장 끝의 [소결] 개수.
    MIN_BULLET_LINES: 4장 전체에서 '- ' 로 시작하는 줄 수.
    MIN_NUMBERED_REFS: REFERENCE 의 '[n] ...' 항목 수(고정 논문 5건이 하한).
    """
    import re

    problems = []

    n_concl = report.count("[소결]")
    if n_concl < MIN_SUMMARY_BLOCKS:
        problems.append(
            f"{FORMAT_PREFIX}[소결] 블록이 {n_concl}개뿐이다. "
            f"4.1/4.2/4.3 각 장 끝에 하나씩 총 {MIN_SUMMARY_BLOCKS}개를 넣어라."
        )

    bullets = sum(
        1
        for t, body in chapters.items()
        if any(k in t for k in PERSPECTIVE_CHAPTERS)
        for line in body.split("\n")
        if line.lstrip().startswith("- ")
    )
    if bullets < MIN_BULLET_LINES:
        problems.append(
            f"{FORMAT_PREFIX}관점별 평가가 개조식이 아니다("
            f"'- ' 로 시작하는 줄 {bullets}개, {MIN_BULLET_LINES}개 이상 필요). "
            "4장의 줄글 문단을 전부 'TurboQuant: / InfiniGen: / →' 3줄 개조식으로 바꿔라."
        )

    ref_body = "\n".join(b for t, b in chapters.items() if "REFERENCE" in t)
    numbered = len(re.findall(r"^\s*\[\d+\]", ref_body, re.M))
    if numbered < MIN_NUMBERED_REFS:
        problems.append(
            f"{FORMAT_PREFIX}REFERENCE에 번호 항목이 {numbered}개뿐이다. "
            f'\'[1] 저자, "제목," 학회/저널, 연도. URL\' 형태로 '
            f"{MIN_NUMBERED_REFS}개 이상 적어라(프롬프트의 확정 논문 5건을 그대로 복사)."
        )

    return problems


def _split_chapters(report):
    """마크다운 헤더 기준으로 보고서를 {장 제목: 본문}으로 자른다."""
    import re

    chapters = {}
    title, buf = "(머리말)", []
    for line in report.split("\n"):
        m = re.match(r"^\s{0,3}#{1,6}\s+(.+?)\s*$", line)
        if m:
            chapters[title] = "\n".join(buf)
            title, buf = m.group(1), []
        else:
            buf.append(line)
    chapters[title] = "\n".join(buf)
    return chapters


def node_h_validate(state):
    report = state.get("final_report", "")
    retry_count = state.get("retry_count", 0)
    chapters = _split_chapters(report)
    titles = list(chapters.keys())

    # 장 제목에서 찾는다. 본문에 "시장"이라는 낱말이 있다고 장이 있는 건 아니다.
    missing = [c for c in REQUIRED_CHAPTERS if not any(c in t for t in titles)]
    missing += _check_format(report, chapters)

    # 서열어는 지정한 장 안에서만 본다.
    scoped = "\n".join(
        body
        for t, body in chapters.items()
        if any(k in t for k in RANKING_CHECK_CHAPTERS)
    )
    forbidden_found = [w for w in FORBIDDEN_WORDS if w in scoped]
    if forbidden_found:
        missing.append(f"서열 표현 발견: {forbidden_found}")

    if not missing:
        result = ValidationResult(is_valid=True, missing_items=[], forced_pass=False)
        return {"validation_result": result.model_dump(), "retry_count": retry_count}

    new_retry_count = retry_count + 1
    if new_retry_count > config.MAX_RETRY_H:
        # 상한 도달. 통과시키되 검증 실패 사실을 보고서에 남긴다.
        # 이게 없으면 검증에 실패한 보고서가 통과 표시로 제출물이 되고,
        # 콘솔 print 말고는 어디에도 흔적이 남지 않는다.
        result = ValidationResult(is_valid=True, missing_items=missing, forced_pass=True)
        return {
            "validation_result": result.model_dump(),
            "retry_count": new_retry_count,
            "final_report": _append_audit_note(report, missing, new_retry_count),
        }

    result = ValidationResult(is_valid=False, missing_items=missing, forced_pass=False)
    return {"validation_result": result.model_dump(), "retry_count": new_retry_count}


def _append_audit_note(report, missing, retry_count):
    """forced_pass 시 검증 기록을 「한계점」 장 뒤(REFERENCE 앞)에 끼워 넣는다."""
    import re

    note = (
        "\n\n### 보고서 검증 기록 (자동 생성)\n\n"
        f"검증 노드(H)가 재작성 상한({config.MAX_RETRY_H}회)에 도달해 "
        f"아래 항목을 충족하지 못한 채 통과 처리했다. 재작성 시도 {retry_count}회.\n\n"
        + "\n".join(f"- {m}" for m in missing)
        + "\n"
    )
    for line in report.split("\n"):
        if re.match(r"^\s{0,3}#{1,6}\s+.*REFERENCE", line):
            return report.replace(line, note.strip() + "\n\n" + line, 1)
    return report + note


def route_after_h(state):
    """H 다음 조건부 엣지. graph.py의 add_conditional_edges가 이 함수를 쓴다."""
    validation = state.get("validation_result", {})
    if validation.get("is_valid", False):
        return "END"
    return "G"

## 5. 배선 테스트 — API 키 없이

H의 재시도 루프(최초 1회 + 재시도 2회 = 총 3회, 상한 도달 시 강제통과)가 제대로 도는지까지 확인한다.

In [ ]:
class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

class FakeMessage:
    def __init__(self, content):
        self.content = content

# make_node_f 는 schemas.Synthesis 가 아니라 create_model 로 만든
# SynthesisStrict 를 넘긴다. 받은 schema_cls 를 그대로 써야 labels 가
# 모델 객체로 생겨 node_f 안의 .model_dump() 가 동작한다.
class FakeLLM_F:
    def with_structured_output(self, schema_cls):
        return FakeStructuredLLM(schema_cls(
            labels={"market": "조건 의존", "stakeholder": "조건 의존",
                    "domain": "조건 의존", "trl": "판단보류"},
            conflicts=[], reasoning="가짜 이유",
        ))

node_f = make_node_f(FakeLLM_F())
f_result = node_f({})
assert "conflicts" in f_result["synthesis"]
assert set(f_result["synthesis"]["labels"]) == {"market", "stakeholder", "domain", "trl"}
print("F 배선 OK:", f_result["synthesis"]["labels"])


def fake_report(with_domain, formatted):
    """H 의 검사 항목(장 유무 / 개조식 / [소결] / 번호 인용)을 골라 맞춘 가짜 보고서."""
    titles = ["시장", "이해관계자"] + (["도메인"] if with_domain else [])
    out = ["# SUMMARY", "내용"]
    for t in titles:
        out.append(f"# {t}")
        if formatted:
            out += [f"- 항목{i}: TurboQuant / InfiniGen 나란히 기록" for i in range(1, 6)]
            out.append(f"[소결] {t} 관점은 조건에 따라 갈린다.")
        else:
            out.append("줄글 문단이라 개조식이 아니다.")
    out.append("# REFERENCE")
    if formatted:
        out += [f'[{i}] 저자{i}, "제목{i}," 학회, 2024. https://example.com/{i}'
                for i in range(1, 6)]
    else:
        out.append("URL 나열")
    return "\n".join(out)


# G 가 매번 다른 보고서를 내도록 해서 H->G 재시도 루프를 직접 확인한다.
# 1차: 도메인 장 누락 + 형식 위반 / 2차: 둘 다 충족
call_count = {"n": 0}
prompts_seen = []

class FakeLLM_G:
    def invoke(self, prompt):
        call_count["n"] += 1
        prompts_seen.append(prompt)
        first = call_count["n"] == 1
        return FakeMessage(fake_report(with_domain=not first, formatted=not first))

node_g = make_node_g(FakeLLM_G())

state = {}
state.update(node_g(state))
state.update(node_h_validate(state))
assert route_after_h(state) == "G"
missing = state["validation_result"]["missing_items"]
assert "도메인" in missing, missing
assert any(m.startswith(FORMAT_PREFIX) for m in missing), missing
print("1차 검증:", missing)

state.update(node_g(state))
state.update(node_h_validate(state))
assert route_after_h(state) == "END", state["validation_result"]
# G 가 형식 위반을 재작성 지시로 받아 갔는지 확인한다(make_node_g 의 formats 분기).
assert "형식 규칙을 어겼다" in prompts_seen[1]
print("2차 검증:", state["validation_result"])
print("G/H 배선 OK, retry_count =", state["retry_count"])

## 6. 실제 LLM 테스트 (F만 — G/H는 최종 통합 노트북에서 실제 데이터로 보는 게 더 의미 있다)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY"):
    from langchain.chat_models import init_chat_model

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    node_f_real = make_node_f(real_llm)
    sample_state = {
        "tech_research": {"TurboQuant": {"trl_assessment": {"trl_ondevice": 4}}, "InfiniGen": {"trl_assessment": {"trl_ondevice": 3}}},
        "market_eval": {"label": "조건 의존"},
        "stakeholder_eval": {"label": "조건 의존"},
        "domain_eval": {"label": "조건 의존"},
    }
    print(node_f_real(sample_state)["synthesis"])
else:
    print("API 키 없음 - 이 셀은 건너뜀.")

## 7. 파일로 저장

In [ ]:
import inspect

TARGET = "../src/nodes_fgh.py"

# 심볼 유실 감지. 아래 parts 목록은 하드코딩이라, 누가 src/nodes_fgh.py 을 직접 고쳐
# 함수·상수를 더해 놓으면 저장하는 순간 그게 조용히 사라진다(2026-09-22 실제 발생).
# 사라진 이름이 있으면 여기서 알린다 - 에러가 안 나서 안 보이는 게 진짜 위험이다.
def _symbols(path):
    import ast, os
    if not os.path.exists(path):
        return set()
    out = set()
    for n in ast.parse(open(path, encoding="utf-8").read()).body:
        if isinstance(n, ast.FunctionDef):
            out.add(n.name)
        elif isinstance(n, ast.Assign):
            out |= {t.id for t in n.targets if isinstance(t, ast.Name)}
    return out


def _src(obj):
    """getsource 결과의 꼬리 개행을 없앤다. 셀 마지막에 있는 함수는 개행이
    안 붙어 나와서, 그대로 이으면 최상위 정의 사이가 빈 줄 1개가 된다."""
    return inspect.getsource(obj).rstrip("\n")


_before = _symbols(TARGET)

q3 = chr(34) * 3
HEADER = (
    q3 + "F, G, H. 평가종합/보고서생성/보고서검증 노드 - 04_agent_FGH_synthesis_report_validate.ipynb에서 생성됨.\n"
    "이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n"
    "from src import config, prompts\n"
    "from src.schemas import Label, ValidationResult\n\n\n"
)

# 상수는 getsource 가 안 되니 값만 찍는다. 그래서 셀의 주석은 .py 로 못 넘어간다 -
# 뜻과 근거는 _check_format 독스트링에 적어 두었다.
CONSTS = (
    f"REQUIRED_CHAPTERS = {REQUIRED_CHAPTERS!r}\n"
    f"FORBIDDEN_WORDS = {FORBIDDEN_WORDS!r}\n"
    f"RANKING_CHECK_CHAPTERS = {RANKING_CHECK_CHAPTERS!r}\n"
    f"\nPERSPECTIVE_CHAPTERS = {PERSPECTIVE_CHAPTERS!r}\n"
    f"MIN_SUMMARY_BLOCKS = {MIN_SUMMARY_BLOCKS!r}\n"
    f"MIN_BULLET_LINES = {MIN_BULLET_LINES!r}\n"
    f"MIN_NUMBERED_REFS = {MIN_NUMBERED_REFS!r}\n"
    f"FORMAT_PREFIX = {FORMAT_PREFIX!r}"
)

parts = [
    _src(make_node_f),
    _src(make_node_g),
    CONSTS,
    _src(_check_format),
    _src(_split_chapters),
    _src(node_h_validate),
    _src(_append_audit_note),
    _src(route_after_h),
]

with open(TARGET, "w", encoding="utf-8") as f:
    f.write(HEADER + "\n\n\n".join(parts) + "\n")   # 정의 사이는 빈 줄 2개(PEP8)

_lost = sorted(_before - _symbols(TARGET))
if _lost:
    print(f"🔴 이번 저장으로 {TARGET} 에서 사라진 심볼: {_lost}")
    print("   노트북이 .py 보다 낡았다는 뜻이다. git diff 로 확인하고,")
    print("   의도한 삭제가 아니면 git checkout 으로 되돌린 뒤 노트북부터 맞출 것.")
else:
    print(f"{TARGET} 저장 완료 (심볼 유실 없음)")